In [1]:
%pip install pandas numpy scikit-learn lightgbm matplotlib

  Using cached lightgbm-4.7.0-py3-none-macosx_12_0_arm64.whl.metadata (18 kB)
Using cached lightgbm-4.7.0-py3-none-macosx_12_0_arm64.whl (1.5 MB)

[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas, numpy, sklearn, matplotlib
from sklearn.ensemble import HistGradientBoostingClassifier
print("All good!")

All good!


In [3]:
import json, glob

files = glob.glob("data/t20s_male_json/*.json")
print(len(files))

with open(files[0]) as f:
    m = json.load(f)

print(m["info"].keys())
print(m["info"]["outcome"])
print(m["innings"][1]["team"], m["innings"][1].get("target"))
print(m["innings"][1]["overs"][0]["deliveries"][0])

3649
dict_keys(['balls_per_over', 'city', 'dates', 'event', 'gender', 'match_type', 'match_type_number', 'officials', 'outcome', 'overs', 'player_of_match', 'players', 'registry', 'season', 'team_type', 'teams', 'toss', 'venue'])
{'result': 'tie', 'eliminator': 'India'}
Sri Lanka {'overs': 20, 'runs': 138}
{'actual_delivery': '0.1', 'batter': 'P Nissanka', 'bowler': 'KK Ahmed', 'non_striker': 'BKG Mendis', 'runs': {'batter': 0, 'extras': 0, 'total': 0}}


In [4]:
import pandas as pd

def parse_match(path, comp):
    with open(path) as f:
        m = json.load(f)
    info = m["info"]
    out = info["outcome"]

    if "winner" not in out or len(m["innings"]) < 2:
        return []

    inn1, inn2 = m["innings"][0], m["innings"][1]
    target = inn2.get("target", {})
    target_runs = target.get("runs")
    target_overs = target.get("overs", info.get("overs", 20))
    if target_runs is None:
        target_runs = sum(d["runs"]["total"] for o in inn1["overs"] for d in o["deliveries"]) + 1

    chasing = inn2["team"]
    won = int(out["winner"] == chasing)
    dls = "method" in out
    total_balls = int(target_overs) * 6 + round((target_overs % 1) * 10)

    rows, runs, wkts, legal = [], 0, 0, 0
    for o in inn2["overs"]:
        for d in o["deliveries"]:
            runs += d["runs"]["total"]
            wkts += len(d.get("wickets", []))
            ex = d.get("extras", {})
            if "wides" not in ex and "noballs" not in ex:
                legal += 1
            rows.append({
                "match_id": path.split("/")[-1][:-5],
                "comp": comp,
                "date": info["dates"][0],
                "target": target_runs,
                "runs": runs,
                "wickets": wkts,
                "balls_left": max(total_balls - legal, 0),
                "runs_needed": target_runs - runs,
                "dls": dls,
                "won": won,
            })
    return rows

all_rows = []
for comp, folder in [("t20i", "t20s_male_json"), ("ipl", "ipl_male_json")]:
    for p in glob.glob(f"data/{folder}/*.json"):
        all_rows += parse_match(p, comp)

df = pd.DataFrame(all_rows)
print(df.shape)
df.head()

(514444, 10)


,match_id,comp,date,target,runs,wickets,balls_left,runs_needed,dls,won
0,1339605,t20i,2023-03-26,259,4,0,119,255,False,1
1,1339605,t20i,2023-03-26,259,8,0,118,251,False,1
2,1339605,t20i,2023-03-26,259,8,0,117,251,False,1
3,1339605,t20i,2023-03-26,259,8,0,116,251,False,1
4,1339605,t20i,2023-03-26,259,9,0,115,250,False,1


In [5]:
print(df.groupby("comp")["match_id"].nunique())
print(df.groupby("comp").apply(lambda g: g.drop_duplicates("match_id")["won"].mean()))
print(df.drop_duplicates("match_id")["dls"].sum(), "DLS matches")
print(df["balls_left"].min(), df["balls_left"].max())
df.to_csv("data/chases.csv", index=False)

comp
ipl     1218
t20i    3529
Name: match_id, dtype: int64
comp
ipl     0.546798
t20i    0.507793
dtype: float64
149 DLS matches
0 120
